# Unsupervised Learning Models

## PCA

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def pca_analysis(
    X_train,
    X_test,
    y_train=None,
    y_test=None,
    n_components=None,
    scale=True,
    random_state=42
):
    """
    Principal Component Analysis (PCA) Utility

    - Fits PCA ONLY on training data (no leakage)
    - Optional feature scaling
    - Transforms train and test sets
    - Prints explained variance ratios
    - Visualizes PC1 vs PC2
    - Supports optional class coloring (if y is provided)

    Parameters
    ----------
    X_train : array-like (n_samples, n_features)
        Training features
    X_test : array-like (n_samples, n_features)
        Test features
    y_train : array-like, optional
        Training labels (for coloring)
    y_test : array-like, optional
        Test labels (for coloring)
    n_components : int or None
        Number of principal components
        If None → keeps all components
    scale : bool, default=True
        Whether to apply StandardScaler
    random_state : int

    Returns
    -------
    pca : fitted PCA object
    X_train_pca : PCA-transformed training data
    X_test_pca : PCA-transformed test data
    """

    # -------------------------------
    # 1. Feature Scaling (IMPORTANT)
    # -------------------------------
    if scale:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
    else:
        X_train_scaled = X_train
        X_test_scaled = X_test

    # -------------------------------
    # 2. PCA Fitting (TRAIN ONLY)
    # -------------------------------
    pca = PCA(n_components=n_components, random_state=random_state)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    # -------------------------------
    # 3. Explained Variance
    # -------------------------------
    print("\nExplained Variance Ratio:")
    for i, var in enumerate(pca.explained_variance_ratio_):
        print(f"PC{i+1}: {var:.4f}")

    print(f"\nTotal Explained Variance: {np.sum(pca.explained_variance_ratio_):.4f}")

    # -------------------------------
    # 4. Visualization: PC1 vs PC2
    # -------------------------------
    if X_train_pca.shape[1] >= 2:
        plt.figure(figsize=(12, 5))

        # ---- Train Plot ----
        plt.subplot(1, 2, 1)
        if y_train is not None:
            scatter = plt.scatter(
                X_train_pca[:, 0],
                X_train_pca[:, 1],
                c=y_train,
                cmap="viridis",
                alpha=0.7
            )
            plt.legend(*scatter.legend_elements(), title="Classes")
        else:
            plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], alpha=0.7)

        plt.xlabel("Principal Component 1")
        plt.ylabel("Principal Component 2")
        plt.title("Train Data: PC1 vs PC2")

        # ---- Test Plot ----
        plt.subplot(1, 2, 2)
        if y_test is not None:
            scatter = plt.scatter(
                X_test_pca[:, 0],
                X_test_pca[:, 1],
                c=y_test,
                cmap="viridis",
                alpha=0.7
            )
            plt.legend(*scatter.legend_elements(), title="Classes")
        else:
            plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], alpha=0.7)

        plt.xlabel("Principal Component 1")
        plt.ylabel("Principal Component 2")
        plt.title("Test Data: PC1 vs PC2")

        plt.tight_layout()
        plt.show()

    else:
        print("Visualization skipped: Need at least 2 PCA components")

    return pca, X_train_pca, X_test_pca


## KNN

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from kneed import KneeLocator
from sklearn.decomposition import PCA

def K_means(
    X,
    k_min=2,
    k_max=20,
    init="k-means++",
    n_init=10,
    max_iter=300,
    scale=True,
    random_state=42,
    plot_pca=True
):
    """
    Automatically determines the optimal number of clusters for K-Means using:
    1. Elbow Method (WCSS + KneeLocator)
    2. Silhouette Score validation

    Parameters
    ----------
    X : array-like (n_samples, n_features)
        Input feature matrix
    k_min : int, default=2
        Minimum number of clusters
    k_max : int, default=20
        Maximum number of clusters
    scale : bool, default=True
        Whether to apply StandardScaler
    random_state : int, default=42
        Random seed

    Returns
    -------
    best_k : int
        Selected optimal number of clusters
    """

    # -------------------- Scaling --------------------
    if scale:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = X

    wcss = []
    silhouette_scores = []
    K = range(1, k_max + 1)

    # -------------------- Compute WCSS & Silhouette --------------------
    for k in K:
        kmeans = KMeans(
            n_clusters=k,
            init=init,
            n_init=n_init,
            max_iter=max_iter,
            random_state=random_state
        )
        labels = kmeans.fit_predict(X_scaled)
        wcss.append(kmeans.inertia_)

        if k >= k_min:
            sil = silhouette_score(X_scaled, labels)
            silhouette_scores.append(sil)
        else:
            silhouette_scores.append(np.nan)

    # -------------------- Elbow Detection --------------------
    knee = KneeLocator(
        K,
        wcss,
        curve="convex",
        direction="decreasing"
    )

    elbow_k = knee.elbow

    # -------------------- Best Silhouette --------------------
    best_sil_k = np.nanargmax(silhouette_scores) + 1
    best_sil_score = np.nanmax(silhouette_scores)

    # -------------------- Final K Selection --------------------
    best_k = elbow_k if elbow_k is not None else best_sil_k

    print("Optimal K Selection Summary")
    print("-" * 45)
    print(f"Elbow Method K       : {elbow_k}")
    print(f"Best Silhouette K    : {best_sil_k}")
    print(f"Best Silhouette Score: {best_sil_score:.4f}")
    print(f"Selected K           : {best_k}")
    print(f"WCSS: {wcss}")
    print(f"Silhouette ScoreS: {silhouette_scores}")
    
    # -------------------- Plot WCSS (Elbow) --------------------
    plt.figure(figsize=(8, 5))
    plt.plot(K, wcss, marker="o")
    if elbow_k:
        plt.axvline(elbow_k, linestyle="--", label=f"Elbow at K={elbow_k}")
    plt.xlabel("Number of Clusters (K)")
    plt.ylabel("WCSS (Inertia)")
    plt.title("Elbow Method for Optimal K")
    plt.legend()
    plt.grid(True)
    plt.show()

    # -------------------- Plot Silhouette --------------------
    plt.figure(figsize=(8, 5))
    plt.plot(K, silhouette_scores, marker="o", color="green")
    plt.axvline(best_sil_k, linestyle="--", label=f"Best K={best_sil_k}")
    plt.xlabel("Number of Clusters (K)")
    plt.ylabel("Silhouette Score")
    plt.title("Silhouette Score vs K")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    best_model= model = KMeans(
        n_clusters=best_k,
        init=init,
        n_init=n_init,
        max_iter=max_iter,
        random_state=random_state
    )
    labels=best_model.fit_predict(X_scaled)
    
    # -------------------- PCA Visualization --------------------
    if plot_pca:
        
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)

        plt.figure(figsize=(8, 6))
        scatter = plt.scatter(
            X_pca[:, 0],
            X_pca[:, 1],
            c=labels,
            cmap="viridis",
            alpha=0.7
        )
        plt.xlabel("Principal Component 1")
        plt.ylabel("Principal Component 2")
        plt.title("K-Means Clustering (PCA Projection)")
        plt.colorbar(scatter, label="Cluster")
        plt.grid(True)
        plt.show()


    return best_model,labels


## Hierarchial Clustering

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.cluster.hierarchy import fcluster


def hierarchical_clustering(
    X,
    n_clusters=None,
    linkage_method="ward",
    metric="euclidean",
    scale=True,
    pca_transform=False,
    pca_components=0.95,
    k_min=2,
    k_max=10,
    plot_dendrogram=True,
    plot_pca=True
):
    """
    Enhanced Hierarchical (Agglomerative) Clustering Utility

    Features:
    - Optional scaling
    - Optional PCA before clustering
    - Automatic cluster selection if n_clusters=None
    - Dendrogram-based selection + silhouette validation
    - PCA visualization

    Parameters
    ----------
    X : array-like (n_samples, n_features)
    n_clusters : int or None
        Number of clusters (auto-selected if None)
    linkage_method : {'ward', 'complete', 'average', 'single'}
    metric : str
        Distance metric (ignored if linkage='ward')
    scale : bool
        Whether to apply StandardScaler
    pca_transform : True or False
        Apply PCA before clustering
    pca_components : float
        Variance to be captured by PCA components if enabled
    k_min : int
        Minimum clusters for silhouette validation
    k_max : int
        Maximum clusters for silhouette validation
    plot_dendrogram : bool
        Plot dendrogram
    plot_pca : bool
        Plot PCA visualization (final clusters)

    Returns
    -------
    model : AgglomerativeClustering
    labels : ndarray
    """

    # -------------------- Scaling --------------------
    if scale:
        scaler = StandardScaler()
        X_processed = scaler.fit_transform(X)
    else:
        X_processed = X

    # -------------------- PCA Transform --------------------
    if pca_transform == True:
        pca = PCA(n_components=pca_components)
        X_processed = pca.fit_transform(X_processed)
        print(f"PCA applied with {pca.n_components_} components")

    # -------------------- Linkage Matrix --------------------
    Z = linkage(
        X_processed,
        method=linkage_method,
        metric=metric if linkage_method != "ward" else "euclidean"
    )

    # -------------------- Dendrogram --------------------
    if plot_dendrogram:
        plt.figure(figsize=(10, 6))
        dendrogram(Z)
        plt.title("Hierarchical Clustering Dendrogram")
        plt.xlabel("Samples")
        plt.ylabel("Distance")
        plt.grid(True)
        plt.show()

    # -------------------- Auto Cluster Selection --------------------
    if n_clusters is None:
        silhouette_scores = {}

        for k in range(k_min, min(k_max, len(X_processed)) + 1):
            labels_k = fcluster(Z, k, criterion="maxclust")
            sil = silhouette_score(X_processed, labels_k)
            silhouette_scores[k] = sil

        n_clusters = max(silhouette_scores, key=silhouette_scores.get)

        print("Automatic Cluster Selection")
        print("-" * 40)
        print(f"Best n_clusters : {n_clusters}")
        print(f"Best silhouette : {silhouette_scores[n_clusters]:.4f}")

        # Plot silhouette vs K
        plt.figure(figsize=(8, 5))
        plt.plot(
            list(silhouette_scores.keys()),
            list(silhouette_scores.values()),
            marker="o"
        )
        plt.axvline(n_clusters, linestyle="--", label=f"Best K = {n_clusters}")
        plt.xlabel("Number of Clusters (K)")
        plt.ylabel("Silhouette Score")
        plt.title("Silhouette Score vs Number of Clusters")
        plt.legend()
        plt.grid(True)
        plt.show()

    # -------------------- Final Model --------------------
    model = AgglomerativeClustering(
        n_clusters=n_clusters,
        linkage=linkage_method,
        metric=metric if linkage_method != "ward" else "euclidean"
    )

    labels = model.fit_predict(X_processed)

    # -------------------- Final Silhouette --------------------
    final_sil = silhouette_score(X_processed, labels)

    print("Hierarchical Clustering Results")
    print("-" * 45)
    print(f"Clusters              : {n_clusters}")
    print(f"Linkage               : {linkage_method}")
    print(f"PCA Applied           : {pca_transform}")
    print(f"Final Silhouette      : {final_sil:.4f}")

    # -------------------- PCA Visualization --------------------
    if plot_pca and X_processed.shape[1] >= 2:
        if pca_transform==False or X_processed.shape[1] < 1:
            pca = PCA(n_components=2)
            X_processed = pca.fit_transform(X_processed)
            
        plt.figure(figsize=(8, 6))
        scatter = plt.scatter(
            X_processed[:, 0],
            X_processed[:, 1],
            c=labels,
            cmap="viridis",
            alpha=0.7
        )
        plt.xlabel("Component 1")
        plt.ylabel("Component 2")
        plt.title("Hierarchical Clustering Visualization")
        plt.colorbar(scatter, label="Cluster")
        plt.grid(True)
        plt.show()

    return model, labels


## DBSCAN Clustering 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from kneed import KneeLocator


def dbscan_clustering(
    X,
    eps=None,
    min_samples=5,
    scale=True,
    pca_transform=False,
    pca_components=2,
    plot_k_distance=True,
    plot_pca=True
):
    """
    DBSCAN Clustering Utility

    Features:
    - Optional scaling
    - Optional PCA before clustering
    - Automatic eps selection using k-distance + KneeLocator
    - Silhouette evaluation (excluding noise)
    - PCA visualization
    - Noise analysis

    Parameters
    ----------
    X : array-like (n_samples, n_features)
    eps : float or None
        Neighborhood radius (auto-selected if None)
    min_samples : int 
        Usually, min_samples= No of features+ 1,
        Minimum samples to form a dense region.
    scale : bool
        Whether to apply StandardScaler
    pca_transform : True or False
        Apply PCA before clustering
    pca_components : int
        PCA components if enabled
    plot_k_distance : bool
        Plot k-distance graph for eps selection
    plot_pca : bool
        Plot PCA visualization

    Returns
    -------
    model : DBSCAN
    labels : ndarray
    """

    # -------------------- Scaling --------------------
    if scale:
        scaler = StandardScaler()
        X_processed = scaler.fit_transform(X)
    else:
        X_processed = X

    # -------------------- PCA Transform --------------------
    if pca_transform.lower() == "yes":
        pca = PCA(n_components=pca_components)
        X_processed = pca.fit_transform(X_processed)
        print(f"PCA applied with {pca.n_components_} components")

    # -------------------- Automatic EPS Selection --------------------
    if eps is None:
        neigh = NearestNeighbors(n_neighbors=min_samples)
        nbrs = neigh.fit(X_processed)
        distances, _ = nbrs.kneighbors(X_processed)

        # k-distance (sorted)
        k_distances = np.sort(distances[:, -1])

        knee = KneeLocator(
            range(len(k_distances)),
            k_distances,
            curve="convex",
            direction="increasing"
        )

        eps = k_distances[knee.knee] if knee.knee is not None else np.percentile(k_distances, 90)

        print(f"Automatically selected eps: {eps:.4f}")

        if plot_k_distance:
            plt.figure(figsize=(8, 5))
            plt.plot(k_distances)
            if knee.knee:
                plt.axvline(knee.knee, linestyle="--", label="Elbow")
            plt.xlabel("Points (sorted)")
            plt.ylabel("k-distance")
            plt.title("k-Distance Graph (DBSCAN eps selection)")
            plt.legend()
            plt.grid(True)
            plt.show()

    # -------------------- DBSCAN Model --------------------
    model = DBSCAN(
        eps=eps,
        min_samples=min_samples
    )

    labels = model.fit_predict(X_processed)

    # -------------------- Cluster Analysis --------------------
    unique_labels = set(labels)
    n_clusters = len(unique_labels) - (1 if -1 in labels else 0)
    noise_ratio = np.sum(labels == -1) / len(labels)

    print("DBSCAN Clustering Results")
    print("-" * 45)
    print(f"eps                  : {eps:.4f}")
    print(f"min_samples          : {min_samples}")
    print(f"Clusters found       : {n_clusters}")
    print(f"Noise points (%)     : {noise_ratio * 100:.2f}")

    # -------------------- Silhouette Score --------------------
    if n_clusters > 1:
        mask = labels != -1  # exclude noise
        sil = silhouette_score(X_processed[mask], labels[mask])
        print(f"Silhouette Score     : {sil:.4f}")
    else:
        sil = None
        print("Silhouette Score     : Not defined (≤1 cluster)")

    # -------------------- PCA Visualization --------------------
    if plot_pca and X_processed.shape[1] >= 2:
        if pca_transform==False or X_processed.shape[1] < 1:
            pca = PCA(n_components=2)
            X_processed = pca.fit_transform(X_processed)
            
        plt.figure(figsize=(8, 6))
        scatter = plt.scatter(
            X_processed[:, 0],
            X_processed[:, 1],
            c=labels,
            cmap="viridis",
            alpha=0.7
        )
        plt.xlabel("Component 1")
        plt.ylabel("Component 2")
        plt.title("DBSCAN Clustering Visualization")
        plt.colorbar(scatter, label="Cluster (-1 = Noise)")
        plt.grid(True)
        plt.show()

    return model, labels


## Anamoly Detection [Outlier Detection]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from kneed import KneeLocator


def anomaly_detection(
    X,
    method="dbscan",
    scale=True,
    pca_transform=False,
    pca_components=2,
    # --- DBSCAN params ---
    eps=None,
    min_samples=5,
    # --- Isolation Forest params ---
    contamination="auto",
    n_estimators=100,
    random_state=42,
    # --- Visualization ---
    plot_results=True
):
    """
    Anomaly Detection Utility using DBSCAN or Isolation Forest

    Parameters
    ----------
    X : array-like (n_samples, n_features)
    method : {'dbscan', 'isolation_forest'}
        Anomaly detection algorithm
    scale : bool
        Apply StandardScaler
    pca_transform : True or False
        Apply PCA before detection
    pca_components : int
        Number of PCA components
    eps : float or None
        DBSCAN eps (auto-selected if None)
    min_samples : int
        DBSCAN min_samples
    contamination : float or 'auto'
        Expected anomaly proportion (Isolation Forest)
    n_estimators : int
        Number of trees (Isolation Forest)
    random_state : int
        Random seed
    plot_results : bool
        Plot anomaly visualization

    Returns
    -------
    model : fitted estimator
    labels : ndarray
        1 = normal, -1 = anomaly
    anomaly_mask : ndarray (bool)
    """

    # -------------------- Scaling --------------------
    if scale:
        X_proc = StandardScaler().fit_transform(X)
    else:
        X_proc = X

    # -------------------- PCA --------------------
    if pca_transform== True:
        pca=PCA(n_components=pca_components)
        X_proc = pca.fit_transform(X_proc)
        print(f"PCA applied with {pca.n_components_} components")

    # ==================== DBSCAN ====================
    if method.lower() == "dbscan":

        # ---- Automatic eps ----
        if eps is None:
            neigh = NearestNeighbors(n_neighbors=min_samples)
            distances, _ = neigh.fit(X_proc).kneighbors(X_proc)
            k_distances = np.sort(distances[:, -1])

            knee = KneeLocator(
                range(len(k_distances)),
                k_distances,
                curve="convex",
                direction="increasing"
            )

            eps = k_distances[knee.knee] if knee.knee else np.percentile(k_distances, 90)
            print(f"Auto-selected eps: {eps:.4f}")

        model = DBSCAN(eps=eps, min_samples=min_samples)
        labels_raw = model.fit_predict(X_proc)

        # DBSCAN convention
        labels = np.where(labels_raw == -1, -1, 1)
        anomaly_mask = labels == -1

        n_anomalies = np.sum(anomaly_mask)
        print("DBSCAN Anomaly Detection")
        print("-" * 40)
        print(f"eps             : {eps:.4f}")
        print(f"min_samples     : {min_samples}")
        print(f"Anomalies found : {n_anomalies}")

    # ==================== ISOLATION FOREST ====================
    elif method.lower() == "isolation_forest":

        model = IsolationForest(
            n_estimators=n_estimators,
            contamination=contamination,
            random_state=random_state
        )

        labels = model.fit_predict(X_proc)
        anomaly_mask = labels == -1

        n_anomalies = np.sum(anomaly_mask)
        print("Isolation Forest Anomaly Detection")
        print("-" * 40)
        print(f"Estimators      : {n_estimators}")
        print(f"Contamination  : {contamination}")
        print(f"Anomalies found: {n_anomalies}")

    else:
        raise ValueError("method must be 'dbscan' or 'isolation_forest'")

    # -------------------- Visualization --------------------
    if plot_results and X_proc.shape[1] >= 2:
        if pca_transform==False or X_proc.shape[1] < 1:
            pca = PCA(n_components=2)
            X_proc = pca.fit_transform(X_proc)
        plt.figure(figsize=(8, 6))
        plt.scatter(
            X_proc[:, 0],
            X_proc[:, 1],
            c=labels,
            cmap="coolwarm",
            alpha=0.7
        )
        plt.title(f"Anomaly Detection using {method.upper()}")
        plt.xlabel("Component 1")
        plt.ylabel("Component 2")
        plt.colorbar(label="1 = Normal, -1 = Anomaly")
        plt.grid(True)
        plt.show()

    return model, labels, anomaly_mask
